# 📊 Project 7.2 — Running Median on Vibration Stream
**DSA for Mechatronics · Week 07 — Heaps & Priority Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import Counter, defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A vibration sensor on a CNC spindle streams readings in real time.
The control system tracks the **running median** — the median of all readings seen so far —
to detect gradual drift without reacting to individual spikes.

**Two-heap approach (O(log n) per reading, O(1) median query):**
- `lo` — max-heap holding the **smaller half** of readings (negate to use Python's min-heap)
- `hi` — min-heap holding the **larger half** of readings
- Invariant: `len(lo) == len(hi)` or `len(lo) == len(hi) + 1`
- Median = `lo[0]` if sizes differ, else average of tops


## Step 1 — Generate your vibration stream

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
N_READINGS   = random.randint(40, 70)
BASE_LEVEL   = round(random.uniform(2.0, 6.0), 2)
NOISE_STD    = round(random.uniform(0.4, 1.2), 2)
N_SPIKES     = random.randint(3, 7)
SPIKE_MAG    = round(random.uniform(8.0, 20.0), 2)
ALERT_MULT   = round(random.uniform(1.8, 3.0), 1)

stream = []
spike_pos = sorted(random.sample(range(3, N_READINGS-3), N_SPIKES))
for i in range(N_READINGS):
    val = BASE_LEVEL + random.gauss(0, NOISE_STD)
    if i in spike_pos:
        val += SPIKE_MAG * random.uniform(0.85, 1.15)
    stream.append(round(val, 3))

ALERT_THRESHOLD = round(ALERT_MULT * BASE_LEVEL, 2)

print(f"Vibration stream parameters:")
print(f"  Readings      : {N_READINGS}")
print(f"  Base level    : {BASE_LEVEL} mm/s²")
print(f"  Noise std     : {NOISE_STD} mm/s²")
print(f"  Spike count   : {N_SPIKES}  at positions {spike_pos}")
print(f"  Spike mag     : ~{SPIKE_MAG} mm/s²")
print(f"  Alert thresh  : {ALERT_THRESHOLD} mm/s²  ({ALERT_MULT}× base)")
print()
print("  First 15 readings:")
for i in range(min(15, N_READINGS)):
    marker = " ◄ SPIKE" if i in spike_pos else ""
    print(f"  [{i:3d}] {stream[i]:8.3f} mm/s²{marker}")


## Step 2 — Two-heap running median

In [ ]:
class RunningMedian:
    """
    Maintains the running median of a data stream using two heaps.

    lo  = max-heap (smaller half) — stored as negatives for Python min-heap
    hi  = min-heap (upper half)

    Invariant after each add():
      len(lo) == len(hi) or len(lo) == len(hi) + 1

    Median:
      odd total  → -lo[0]          (lo has one extra element)
      even total → (-lo[0] + hi[0]) / 2
    """
    def __init__(self):
        self.lo = []   # max-heap via negation
        self.hi = []   # min-heap

    def add(self, val):
        # Step 1: push to lo (as negative)
        heapq.heappush(self.lo, -val)
        # Step 2: ensure every element in lo ≤ every element in hi
        if self.hi and (-self.lo[0]) > self.hi[0]:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # Step 3: balance sizes so len(lo) == len(hi) or len(lo) == len(hi)+1
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if not self.lo: return None
        if len(self.lo) == len(self.hi):
            return round((-self.lo[0] + self.hi[0]) / 2, 4)
        return round(-self.lo[0], 4)

    def lower_count(self): return len(self.lo)
    def upper_count(self): return len(self.hi)

rm = RunningMedian()
medians, alerts = [], []

print(f"Running median (every 5th reading shown + all spikes):")
print(f"  {'Idx':>4}  {'Reading':>10}  {'Median':>10}  {'lo':>4}  {'hi':>4}  Alert")
print(f"  {'─'*4}  {'─'*10}  {'─'*10}  {'─'*4}  {'─'*4}  {'─'*10}")
for i, val in enumerate(stream):
    rm.add(val)
    med = rm.median()
    medians.append(med)
    alert = med >= ALERT_THRESHOLD
    if alert: alerts.append(i)
    flag = "⚠ ALERT" if alert else ""
    if i % 5 == 0 or i in spike_pos or alert:
        print(f"  {i:4d}  {val:10.3f}  {med:10.4f}  {rm.lower_count():4d}  {rm.upper_count():4d}  {flag}")

final_median = medians[-1]
median_range = round(max(medians) - min(medians), 4)
print(f"\n  Final running median : {final_median} mm/s²")
print(f"  Median range         : {median_range} mm/s²  (max – min over stream)")
print(f"  Alert readings       : {len(alerts)}  at positions {alerts}")


## Step 3 — Compare running median vs simple mean

In [ ]:
# Simple running mean for comparison
means = []
total = 0.0
for i, val in enumerate(stream):
    total += val
    means.append(round(total / (i + 1), 4))

# Compare at spike positions
print("Running median vs running mean at spike positions:")
print(f"  {'Pos':>4}  {'Reading':>10}  {'Median':>10}  {'Mean':>10}  {'|Med-Mean|':>12}")
print(f"  {'─'*4}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*12}")
for pos in spike_pos:
    diff = round(abs(medians[pos] - means[pos]), 4)
    print(f"  {pos:4d}  {stream[pos]:10.3f}  {medians[pos]:10.4f}  {means[pos]:10.4f}  {diff:12.4f}")

# After all spikes
final_mean = means[-1]
robustness = round(abs(final_median - BASE_LEVEL), 4)
mean_drift = round(abs(final_mean  - BASE_LEVEL), 4)

print(f"\n  Base level              : {BASE_LEVEL} mm/s²")
print(f"  Final median deviation  : {robustness} mm/s²  (from base)")
print(f"  Final mean deviation    : {mean_drift} mm/s²  (from base)")
winner = "median" if robustness < mean_drift else "mean"
print(f"  More robust estimator   : {winner}")


## Step 4 — Heap state snapshot

In [ ]:
print(f"Final two-heap state:")
print(f"  lo (smaller half, max-heap): {len(rm.lo)} elements")
print(f"    top (max of lower half)  : {-rm.lo[0]:.4f} mm/s²")
print(f"  hi (upper half, min-heap) : {len(rm.hi)} elements")
if rm.hi:
    print(f"    top (min of upper half)  : {rm.hi[0]:.4f} mm/s²")
print(f"  Median                    : {rm.median()} mm/s²")
print()
# Verify median manually
all_sorted = sorted(stream)
n = len(all_sorted)
true_median = all_sorted[n//2] if n % 2 == 1 else round((all_sorted[n//2-1]+all_sorted[n//2])/2, 4)
match = abs(rm.median() - true_median) < 1e-4
print(f"  Verification (sort & pick): {true_median}")
print(f"  Two-heap result           : {rm.median()}")
print(f"  Match                     : {'✅ YES' if match else '❌ NO'}")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " RUNNING MEDIAN — VIBRATION STREAM REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  STREAM PARAMETERS" + " "*(W-19) + "║")
print(f"║  {'Readings':<26}: {N_READINGS:<26}║")
print(f"║  {'Base level':<26}: {BASE_LEVEL} mm/s²{'':<20}║")
print(f"║  {'Spikes injected':<26}: {N_SPIKES}  at {spike_pos}{'':<4}║"[:W+2]+"║")
print(f"║  {'Alert threshold':<26}: {ALERT_THRESHOLD} mm/s²  ({ALERT_MULT}× base){'':<10}║")
print("╠" + "═"*W + "╣")
print("║  TWO-HEAP RESULTS" + " "*(W-18) + "║")
print(f"║  {'Final running median':<26}: {final_median} mm/s²{'':<18}║")
print(f"║  {'Final running mean':<26}: {final_mean} mm/s²{'':<18}║")
print(f"║  {'Median deviation':<26}: {robustness} mm/s²{'':<18}║")
print(f"║  {'Mean deviation':<26}: {mean_drift} mm/s²{'':<18}║")
print(f"║  {'More robust':<26}: {winner:<26}║")
print(f"║  {'Alert readings':<26}: {len(alerts):<26}║")
print("╠" + "═"*W + "╣")
print("║  VERIFICATION" + " "*(W-14) + "║")
print(f"║  {'True median (sorted)':<26}: {true_median:<26}║")
print(f"║  {'Two-heap median':<26}: {rm.median():<26}║")
print(f"║  {'Verified correct':<26}: {'YES ✅' if match else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which heap concept did you find most important, and why?**
Pick one technique (sift-up, sift-down, two-heap median, heapify…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
